Need to do some further cleaning on the data so will do so here

In [15]:
#libraries
import math
import pandas as pd
import numpy as np

In [12]:
pd.options.mode.chained_assignment = None
#load in csv
df = pd.read_csv('uniqueevilmasterdoc.csv')
#start by removing . because they are rly hard to remove for some reason
dropped = df.drop([1206, 1364, 1365])
#remove @ and get 1593
cleaned_df = dropped[~dropped.spaces.str.contains("@")]
#remove - and get 1583
cleaned_df = cleaned_df[~cleaned_df.spaces.str.contains("-")]
#remove ! and get 1574
cleaned_df = cleaned_df[~cleaned_df.spaces.str.contains("!")]
#remove _ and get 1567
cleaned_df = cleaned_df[~cleaned_df.spaces.str.contains("_")]
#remove / and get 1567
cleaned_df = cleaned_df[~cleaned_df.spaces.str.contains("/")]
#remove & and get 1566
cleaned_df = cleaned_df[~cleaned_df.spaces.str.contains("&")]
#remove + and get 1558
cleaned_df = cleaned_df[~cleaned_df.spaces.str.contains('+', regex = False)]
#remove * and get 1487
cleaned_df = cleaned_df[~cleaned_df.spaces.str.contains("*", regex = False)]
#remove # and get 1486
cleaned_df = cleaned_df[~cleaned_df.spaces.str.contains("#")]
#remove ' and get 1479
cleaned_df = cleaned_df[~cleaned_df.spaces.str.contains("'")]
#remove . and get 1477
cleaned_df = cleaned_df[~cleaned_df.spaces.str.contains(".", regex = False)]

Now putting in the big offensive dataset 

In [22]:
#load the big dataset
off = pd.read_csv('labeled_data.csv')
off.head()
#remove class of 2 
filt = off[off['class'] < 2]
#create hate speech and offensive language scores 
filt['hate_score'] = filt['hate_speech']/filt['count']
filt['off_score'] = filt['offensive_language']/filt['count']
filt.head()

,Unnamed: 0,count,hate_speech,offensive_language,neither,class,tweet,hate_score,off_score
1,1,3,0,3,0,1,!!!!! RT @mleew17: boy dats cold...tyga dwn ba...,0.000000,1.000000
2,2,3,0,3,0,1,!!!!!!! RT @UrKindOfBrand Dawg!!!! RT @80sbaby...,0.000000,1.000000
3,3,3,0,2,1,1,!!!!!!!!! RT @C_G_Anderson: @viva_based she lo...,0.000000,0.666667
4,4,6,0,6,0,1,!!!!!!!!!!!!! RT @ShenikaRoberts: The shit you...,0.000000,1.000000
5,5,3,1,2,0,1,"!!!!!!!!!!!!!!!!!!""@T_Madison_x: The shit just...",0.333333,0.666667


Now counting the amount of times each item appears

In [23]:
#list of offensive words
off_words = list(cleaned_df['nospace'])
#create df of offensive words
off = pd.DataFrame(cleaned_df['nospace'])
tweets = list(filt['tweet'])
#create the new coluns that we will iterate over
off['hate_speech'] = 1
off['offensive_language'] = 1
off['count_words'] = 1

#this gives us the count of each offensive word
for i, value in enumerate(off_words):
    count = sum(value in s for s in tweets)
    off['count_words'][i] = count

In [24]:
#this gives us the score
for i, value in enumerate(off_words):
    newdf = pd.DataFrame(filt[filt['tweet'].str.contains(value)])
    off['hate_speech'][i] = sum(newdf['hate_speech'])
    off['offensive_language'][i] = sum(newdf['offensive_language'])

In [29]:
#find out how many people ranked a tweet with this word in it 
off['total_rating'] = off['hate_speech'] + off['offensive_language']
#find out what percent ranked it hate and offensive out of the total
off['hate_out_tot'] = off['hate_speech']/off['total_rating']
off['offense_out_tot'] = off['offensive_language']/off['total_rating']

#maximum count is 11151
off['count_perc'] = off['count_words']/11151
#get log
off['log_count'] = np.log(off['count_perc'])
off['pos_log'] = off['log_count'].abs()

inf


/Users/naomiliftman/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arraylike.py:397: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


,nospace,hate_speech,offensive_language,count_words,total_rating,hate_out_tot,offense_out_tot,count_perc,log_count,pos_log
0,anal,3,30,10,33,0.090909,0.909091,0.000897,-7.016699,7.016699
1,anus,0,15,4,15,0.000000,1.000000,0.000359,-7.932990,7.932990
2,apeshit,0,0,0,0,NaN,NaN,0.000000,-inf,inf
3,ass,793,5362,1923,6155,0.128838,0.871162,0.172451,-1.757643,1.757643
4,asshole,29,110,44,139,0.208633,0.791367,0.003946,-5.535095,5.535095
5,bangbus,0,0,0,0,NaN,NaN,0.000000,-inf,inf
6,bastard,11,13,8,24,0.458333,0.541667,0.000717,-7.239843,7.239843
7,bbw,0,3,1,3,0.000000,1.000000,0.000090,-9.319284,9.319284
8,bdsm,0,0,0,0,NaN,NaN,0.000000,-inf,inf
9,beaner,42,34,28,76,0.552632,0.447368,0.002511,-5.987080,5.987080


In [38]:
#now get a csv with the cleaned data and count and number of offensives
off.to_csv('master_counts_scores.csv', index=False)